# Is it symmetric? A laminate, step by step

This notebook follows the *Learn* part of the lesson
[Is it symmetric?](https://m-aryayi.github.io/composites-playground/tools/laminate-symmetry/)
of Composites Playground. Each step has a short text and its code.
Run the cells from the top to the bottom. Then change the stack and run them again.

Units: MPa and mm, so [A] is in N/mm, [B] in N, and [D] in N·mm.
Reference: Kaw, A. K. (2005). *Mechanics of Composite Materials* (2nd ed.). CRC Press.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=1, suppress=True)

## 1. Plies and fiber angles

A laminate code lists the ply angles from the top ply to the bottom ply.
The angle goes from the x axis (the pull) to the fibers.

In [ ]:
angles = [0, 45, 90]      # the stack [0/+45/90], top to bottom, in degrees
ply_thickness = 0.125     # mm

print(f"{len(angles)} plies, {len(angles) * ply_thickness} mm thick")

## 2. The stiffness of one ply

In its own axes (1 along the fibers, 2 across them), a ply has the reduced
stiffness matrix [Q] (Kaw Eq. 2.93). The data is graphite/epoxy (Kaw Table 2.1).

In [ ]:
E1, E2, nu12, G12 = 181000.0, 10300.0, 0.28, 7170.0   # MPa

nu21 = nu12 * E2 / E1
denom = 1.0 - nu12 * nu21
Q = np.array([[E1 / denom, nu12 * E2 / denom, 0.0],
              [nu12 * E2 / denom, E2 / denom, 0.0],
              [0.0, 0.0, G12]])
print("[Q] in MPa:")
print(Q)

In the x-y axes of the laminate, a ply at the angle θ has the
transformed stiffness [Q̄] = [Tε]ᵀ [Q] [Tε] (Kaw Eq. 2.104, in matrix form).
At 0° and 90°, Q̄16 = 0. At other angles it is not zero, and it changes sign with θ.

In [ ]:
def strain_transform(theta_deg):
    """Return the matrix that turns x-y strains into fiber-axis strains."""
    c = np.cos(np.radians(theta_deg))
    s = np.sin(np.radians(theta_deg))
    return np.array([[c * c, s * s, c * s],
                     [s * s, c * c, -c * s],
                     [-2 * c * s, 2 * c * s, c * c - s * s]])


def q_bar(theta_deg):
    """Return the stiffness of a ply at theta in the x-y axes, in MPa."""
    t = strain_transform(theta_deg)
    return t.T @ Q @ t


for theta in (0, 30, -30, 90):
    print(f"theta = {theta:4}°:  Qbar16 = {q_bar(theta)[0, 2]:9.1f} MPa")

## 3. From plies to a laminate

Each ply k lies between the heights h(k-1) (top) and h(k) (bottom), measured from
the mid-plane (Kaw Eq. 4.20). The plies add up to [A], [B], and [D]
(Kaw Eq. 4.28).

In [ ]:
def abd_matrices(angles, thickness):
    """Return [A], [B], [D] of a stack; ply 1 is the top ply."""
    n = len(angles)
    h = thickness * (np.arange(n + 1) - n / 2)    # ply surfaces from the mid-plane
    a, b, d = np.zeros((3, 3)), np.zeros((3, 3)), np.zeros((3, 3))
    for k, theta in enumerate(angles):
        qb = q_bar(theta)
        a += qb * (h[k + 1] - h[k])
        b += qb * (h[k + 1] ** 2 - h[k] ** 2) / 2
        d += qb * (h[k + 1] ** 3 - h[k] ** 3) / 3
    return a, b, d


A, B, D = abd_matrices(angles, ply_thickness)
print("[A] in N/mm:", A, "[B] in N:", B, "[D] in N mm:", D, sep="\n")

Compare with the book: Kaw Example 4.2 is [0/30/−45] with 5 mm plies.
The book gives A11 = 1.739e6 N/mm, B11 = −3.129e6 N, and D11 = 3.343e7 N·mm.

In [ ]:
A, B, D = abd_matrices([0, 30, -45], 5.0)
print(f"A11 = {A[0, 0]:.4g} N/mm,  B11 = {B[0, 0]:.4g} N,  D11 = {D[0, 0]:.4g} N mm")

## 4. Symmetry and [B]

In a symmetric stack, each ply above the mid-plane has a mirror ply below it.
Their terms in [B] have opposite signs and cancel, so [B] = 0
(Kaw Section 5.2.1).

In [ ]:
for stack in ([0, 90], [0, 90, 90, 0]):
    _, B, _ = abd_matrices(stack, ply_thickness)
    print(f"{stack}:  largest |B| = {np.abs(B).max():.1f} N")

## 5. Balance and A16, A26

In a balanced stack, each +θ ply has a −θ ply of the same thickness, at any
place. Their Q̄16 and Q̄26 cancel in [A], so A16 = A26 = 0
(Kaw Section 5.2.5).

In [ ]:
for stack in ([45], [45, -45], [0, 45, 90, 90, 45, 0]):
    A, _, _ = abd_matrices(stack, ply_thickness)
    print(f"{stack}:  A16 = {A[0, 2]:.1f},  A26 = {A[1, 2]:.1f} N/mm")

## 6. What a pull does

The pull Nx = 100 N/mm, with no moment, gives the mid-plane strains and the
curvatures from the 6 equations of Kaw Eq. 4.29.

In [ ]:
def pull(angles, nx=100.0):
    """Return the mid-plane strains and curvatures for a pull nx in N/mm."""
    a, b, d = abd_matrices(angles, ply_thickness)
    abd = np.block([[a, b], [b, d]])
    response = np.linalg.solve(abd, [nx, 0.0, 0.0, 0.0, 0.0, 0.0])
    return response[:3], response[3:]


strain, curvature = pull(angles)
print("strains  eps_x, eps_y, gamma_xy:  ", [f"{v:.3e}" for v in strain])
print("curvatures  kx, ky, kxy (1/mm):  ", [f"{v:.3e}" for v in curvature])

The out-of-plane shape of a square plate (100 mm) follows from the
curvatures: w = −(κx x² + κy y² + κxy x y) / 2. The plot shows only the shape:
w is scaled so that its largest value is 1. A flat plate shows as a flat square.

In [ ]:
side = np.linspace(-50.0, 50.0, 21)
x, y = np.meshgrid(side, side)
w = -0.5 * (curvature[0] * x**2 + curvature[1] * y**2 + curvature[2] * x * y)
largest = np.abs(w).max()
shape = w / largest if largest > 1e-12 else np.zeros_like(w)

fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(x, y, shape, cmap="Greens", edgecolor="white", linewidth=0.3)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_zlim(-1, 1)
ax.set_title(f"{angles} pulled along x (shape only)")
plt.show()

## Your turn

1. Set `angles = [0, 90, 90, 0]` in step 1 and run all cells. [B] becomes zero,
   and the plate stays flat.
2. Try `[45, -45]`: balanced (A16 = A26 = 0) but not symmetric. It twists.
3. Try `[0, 45, 0]`: symmetric but not balanced. It stays flat but shears
   (look at gamma_xy).